In [1]:
import pyarrow.parquet as pq
import geopandas as gpd
import pandas as pd
from pathlib import Path
import yaml

In [2]:
project_root = Path.cwd().parents[0]
config_path  = project_root / "configs" / "paths.yaml"

with open(config_path) as f:
    paths = yaml.safe_load(f)

parquet_dir = project_root / paths["data"]["processed"] / "geoparquets"

# --- pick the file to inspect ---
TARGET = "cel_hex_15k.parquet"   # change this to any file in the directory

fp = parquet_dir / TARGET
print(fp)

c:\Users\Zachary\phd_classes\uganda\data\processed\geoparquets\cel_hex_15k.parquet


### Schema and row count (reads only metadata — no data loaded)

In [3]:
pf = pq.ParquetFile(fp)

print(f"Rows      : {pf.metadata.num_rows:,}")
print(f"Row groups: {pf.metadata.num_row_groups}")
print(f"Columns   : {pf.metadata.num_columns}")
print()
print(pf.schema_arrow)

Rows      : 3,851,392
Row groups: 59
Columns   : 34

oid: int32
adm2_en: string
adm1_en: string
adm0_en: string
name: string
lat: double
lon: double
dist_to_road: double
dist_to_market: double
dist_to_rivers: double
dist_to_rivers_and_streams: double
dist_to_rivers_plus: double
dist_to_pop_center_1: double
dist_to_pop_center_2: double
dist_to_pop_center_3: double
dist_to_pop_center_4: double
elev_avg: double
elev_sd: double
slope_avg: double
ag_total_area: double
ag_y_2017_pct: double
ag_y_2018_pct: double
ag_y_2019_pct: double
ag_y_2020_pct: double
ag_y_2021_pct: double
ag_y_2022_pct: double
ag_y_2023_pct: double
ag_y_2024_pct: double
centroid_wkt: string
geom: binary
geom_bbox: struct<xmin: float not null, ymin: float not null, xmax: float not null, ymax: float not null>
  child 0, xmin: float not null
  child 1, ymin: float not null
  child 2, xmax: float not null
  child 3, ymax: float not null
-- schema metadata --
geo: '{"version":"1.1.0","primary_column":"geom","columns":{"geom"

### Column names only

In [4]:
cols = pf.schema_arrow.names
print(f"{len(cols)} columns:")
for c in cols:
    print(" ", c)

31 columns:
  oid
  adm2_en
  adm1_en
  adm0_en
  name
  lat
  lon
  dist_to_road
  dist_to_market
  dist_to_rivers
  dist_to_rivers_and_streams
  dist_to_rivers_plus
  dist_to_pop_center_1
  dist_to_pop_center_2
  dist_to_pop_center_3
  dist_to_pop_center_4
  elev_avg
  elev_sd
  slope_avg
  ag_total_area
  ag_y_2017_pct
  ag_y_2018_pct
  ag_y_2019_pct
  ag_y_2020_pct
  ag_y_2021_pct
  ag_y_2022_pct
  ag_y_2023_pct
  ag_y_2024_pct
  centroid_wkt
  geom
  geom_bbox


### Preview — first N rows (no geometry)

In [6]:
N = 5

# Read only the first batch from the first row group — lightweight
batch = next(pf.iter_batches(batch_size=N))
df = batch.to_pandas()

# Drop geometry for clean display
df.drop(columns=[c for c in df.columns if c == "geom"], errors="ignore").sort_values(by = "oid", ascending = True).head(N)

,oid,adm2_en,adm1_en,adm0_en,name,lat,lon,dist_to_road,dist_to_market,dist_to_rivers,...,ag_y_2017_pct,ag_y_2018_pct,ag_y_2019_pct,ag_y_2020_pct,ag_y_2021_pct,ag_y_2022_pct,ag_y_2023_pct,ag_y_2024_pct,centroid_wkt,geom_bbox
3,100208,NaN,NaN,NaN,NaN,-1.134862,30.457590,9086.916506,67111.539618,1249.613194,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(217055.09085368156 -125560.88830653741),"{'xmin': 216977.515625, 'ymin': -125628.0625, ..."
4,100432,NaN,NaN,NaN,NaN,0.036688,30.458085,5945.702164,99286.306128,6400.446151,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(217055.09085368147 4059.1097196756486),"{'xmin': 216977.515625, 'ymin': 3991.948730468..."
0,1001281,Kiruhura,Western,Uganda,NaN,-0.646434,30.947845,744.265924,9853.034524,5291.766126,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(271611.80071853934 -71496.58861166608),"{'xmin': 271534.25, 'ymin': -71563.75, 'xmax':..."
1,1001300,Kiruhura,Western,Uganda,NaN,-0.623359,30.947854,1752.349482,12302.002584,7633.756547,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(271611.8007185392 -68944.48502358525),"{'xmin': 271534.25, 'ymin': -69011.6484375, 'x..."
2,1001329,Kiruhura,Western,Uganda,NaN,-0.588140,30.947867,1017.712967,16101.778996,11207.001812,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,POINT(271611.80071853905 -65049.16902072494),"{'xmin': 271534.25, 'ymin': -65116.3359375, 'x..."


### Preview — first N rows including geometry (GeoDataFrame)

In [9]:
gdf = gpd.read_parquet(fp)
gdf.shape

(3851392, 30)

### Read a specific subset of columns

In [8]:
COLS = ["oid", "name", "adm2_en", "elev_avg", "slope_avg", "dist_to_road", "geom"]

gdf_slim = gpd.read_parquet(fp, columns=COLS)
gdf_slim.sort_values(by = "oid", ascending=False)

,oid,name,adm2_en,elev_avg,slope_avg,dist_to_road,geom
626654,3851119,NaN,NaN,944.0,3.0,24175.267757,"POLYGON ((546683.087 415820.875, 546644.312 41..."
2042327,3851118,NaN,NaN,943.0,2.0,24184.760991,"POLYGON ((546683.087 415686.554, 546644.312 41..."
2508089,3851117,NaN,NaN,943.0,2.0,24194.996211,"POLYGON ((546683.087 415552.233, 546644.312 41..."
1259488,3851116,NaN,NaN,942.0,2.0,24205.972476,"POLYGON ((546683.087 415417.912, 546644.312 41..."
1958688,3851115,NaN,NaN,943.0,2.0,24217.688778,"POLYGON ((546683.087 415283.591, 546644.312 41..."
...,...,...,...,...,...,...,...
1809993,4,NaN,NaN,935.0,1.0,12051.610287,"POLYGON ((188051.239 22729.762, 188012.464 227..."
2503417,3,NaN,NaN,934.0,2.0,12058.075446,"POLYGON ((188051.239 22595.441, 188012.464 226..."
926718,2,NaN,NaN,934.0,2.0,12066.032522,"POLYGON ((188051.239 22461.12, 188012.464 2252..."
3463485,1,NaN,NaN,937.0,2.0,12075.478564,"POLYGON ((188051.239 22326.799, 188012.464 223..."


### Summary stats on a few columns (reads full column, but only those columns)

In [11]:
STAT_COLS = ["oid","elev_avg", "slope_avg", "dist_to_road"]

# Filter to columns that actually exist
available = set(pf.schema_arrow.names)
stat_cols = [c for c in STAT_COLS if c in available]

df_stats = pq.read_table(fp, columns=stat_cols).to_pandas()
df_stats.describe()

,oid,elev_avg,slope_avg,dist_to_road
count,3.851392e+06,3.851392e+06,3.851392e+06,3.851392e+06
mean,1.925557e+06,1.049337e+03,4.154070e+00,3.115495e+03
std,1.111684e+06,2.610853e+02,4.281628e+00,4.120514e+03
min,0.000000e+00,5.320000e+02,0.000000e+00,8.278555e-04
25%,9.628478e+05,8.660000e+02,2.000000e+00,3.786930e+02
50%,1.925540e+06,1.045000e+03,3.000000e+00,1.330437e+03
75%,2.888271e+06,1.258000e+03,5.000000e+00,4.428129e+03
max,3.851119e+06,2.994000e+03,5.800000e+01,2.994833e+04
